# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

In [ ]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

In [ ]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [ ]:
print(embeddings.shape, embeddings.dtype)

In [ ]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [ ]:
!pip install faiss-cpu

In [ ]:
# código base para FAISS
import faiss
import numpy as np

# Asumiendo `embeddings` en un array NxD
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

query_embedding = query_vec.reshape(1, -1)
D, I = index.search(query_embedding, k=10)

In [ ]:
# Mostrar resultados conceptuales
print(f"Distancias/Similitudes del Top-{10}:", D)
print(f"Índices de los chunks recuperados:", I)

In [ ]:
# Ejemplo del primer resultado conceptual
print("\nPrimer resultado recuperado:")
print(chunks_df.iloc[I[0][0]]["text"])

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [ ]:
!pip install qdrant-client

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# Conectar a una instancia local Qdrant (en memoria)
qdrant_client = QdrantClient(":memory:")
COLLECTION_NAME = "wikipedia_chunks"

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
)

# Inserción de ids, embeddings y payloads (metadata)
points = []
for idx, row in chunks_df.iterrows():
    points.append(
        PointStruct(
            id=int(idx),
            vector=embeddings[idx].tolist(),
            payload={
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"]),
                "text": str(row["text"])
            }
        )
    )

# Subir los puntos por batches para no saturar la memoria
batch_size = 2000
for i in range(0, len(points), batch_size):
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i+batch_size]
    )


In [ ]:
def qdrant_search(client, query_embedding, k=5):
    # Asegura un vector plano unidimensional para la API de Qdrant
    vector_plano = query_embedding.flatten().tolist()

    search_result = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector_plano,
        limit=k
    )

    output = []
    for hit in search_result:
        output.append((
            hit.id,
            hit.score,
            hit.payload["text"],
            {"doc_id": hit.payload["doc_id"], "chunk_id": hit.payload["chunk_id"]}
        ))
    return output

# Ejecución de prueba
results_qdrant = qdrant_search(qdrant_client, query_vec, k=5)
print(f"Recuperados de Qdrant: {len(results_qdrant)} documentos.")
print(results_qdrant[0])

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [ ]:
!pip install pymilvus[milvus_lite]

In [ ]:
from pymilvus import connections, utility, FieldSchema, CollectionSchema, DataType, Collection
from pymilvus import MilvusClient
# 1. Conectar a Milvus Lite (en memoria, ideal para Jupyter Notebooks)
connections.connect("default", uri="./milvus_demo.db")

COLLECTION_MILVUS = "wikipedia_milvus"

# Eliminar si ya existe para evitar colisiones
if utility.has_collection(COLLECTION_MILVUS):
    utility.drop_collection(COLLECTION_MILVUS)

In [ ]:
# Esquema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=embeddings.shape[1]),
    FieldSchema(name="doc_id", dtype=DataType.INT64),
    FieldSchema(name="chunk_id", dtype=DataType.INT64),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1500)
]
schema = CollectionSchema(fields, description="Wikipedia Chunks")
collection = Collection(name=COLLECTION_MILVUS, schema=schema)

In [ ]:
# Inserta datos organizados por columnas/campos
# data = [
#     chunks_df.index.tolist(),                       # id
#     embeddings.tolist(),                            # embedding
#     chunks_df["doc_id"].astype(int).tolist(),       # doc_id
#     chunks_df["chunk_id"].astype(int).tolist(),     # chunk_id
#     chunks_df["text"].astype(str).tolist()          # text
# ]
# collection.insert(data)

# Fix: Insert data in batches to avoid RESOURCE_EXHAUSTED error
batch_size = 1000 # Adjust batch size as needed

# Prepare data for batch insertion
ids = chunks_df.index.tolist()
embeddings_list = embeddings.tolist()
doc_ids = chunks_df["doc_id"].astype(int).tolist()
chunk_ids = chunks_df["chunk_id"].astype(int).tolist()
texts = chunks_df["text"].astype(str).tolist()

for i in range(0, len(ids), batch_size):
    batch_ids = ids[i:i + batch_size]
    batch_embeddings = embeddings_list[i:i + batch_size]
    batch_doc_ids = doc_ids[i:i + batch_size]
    batch_chunk_ids = chunk_ids[i:i + batch_size]
    batch_texts = texts[i:i + batch_size]

    batch_data = [
        batch_ids,
        batch_embeddings,
        batch_doc_ids,
        batch_chunk_ids,
        batch_texts
    ]
    collection.insert(batch_data)

collection.flush()

In [ ]:
def milvus_search(query_embedding: np.ndarray, k: int = 5, search_params=None):
    if search_params is None:
        search_params = {"metric_type": "COSINE", "params": {"ef": 64}}

    results = collection.search(
        data=query_embedding.reshape(1, -1).tolist(),
        anns_field="embedding",
        param=search_params,
        limit=k,
        output_fields=["doc_id", "chunk_id", "text"]
    )

    output = []
    for hits in results:
        for hit in hits:
            output.append((
                hit.id,
                hit.score,
                hit.entity.get("text"),
                {"doc_id": hit.entity.get("doc_id"), "chunk_id": hit.entity.get("chunk_id")}
            ))
    return output

In [ ]:
# Mini experimento: k=5 vs k=20
import time

for current_k in [5, 20]:
    start_time = time.time()
    res_milvus = milvus_search(query_vec, k=current_k)
    duration = time.time() - start_time
    print(f"Resultados para k={current_k} (Tiempo de consulta: {duration:.5f} seg): Total devueltos {len(res_milvus)}")

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [ ]:
!pip install "weaviate-client<4"


In [ ]:
import weaviate

# Conectar a Weaviate embebido (servidor local en memoria, sin Docker)
weaviate_client = weaviate.Client(embedded_options=weaviate.embedded.EmbeddedOptions())

# Definir un esquema (Clase). Si ya existe (por ejemplo, de una ejecución anterior),
# lo eliminamos primero para evitar un error de "clase duplicada".
if weaviate_client.schema.exists("Document"):
    weaviate_client.schema.delete_class("Document")

class_obj = {
    "class": "Document",
    "description": "Wikipedia Chunks con Embeddings E5",
    "vectorizer": "none",  # Nosotros proveemos nuestros propios vectores externos
    "properties": [
        {"name": "text", "dataType": ["text"]},
        {"name": "doc_id", "dataType": ["int"]},
        {"name": "chunk_id", "dataType": ["int"]}
    ]
}

weaviate_client.schema.create_class(class_obj)


In [ ]:
# Insertar objetos usando procesamiento Batch eficiente
with weaviate_client.batch(batch_size=100) as batch:
    for idx, row in chunks_df.iterrows():
        properties = {
            "text": str(row["text"]),
            "doc_id": int(row["doc_id"]),
            "chunk_id": int(row["chunk_id"])
        }
        batch.add_data_object(
            data_object=properties,
            class_name="Document",
            vector=embeddings[idx].tolist()
        )


In [ ]:
def weaviate_search(query_embedding: np.ndarray, k: int = 5):
    near_vector = {"vector": query_embedding.flatten().tolist()}

    result = (
        weaviate_client.query
        .get("Document", ["text", "doc_id", "chunk_id"])
        .with_near_vector(near_vector)
        .with_limit(k)
        .with_additional(["id", "distance"])
        .do()
    )

    hits = result["data"]["Get"]["Document"]
    output = []
    for h in hits:
        # Weaviate devuelve distancias en su formato interno según la versión
        score = h["_additional"]["distance"]
        output.append((
            h["_additional"]["id"],
            score,
            h["text"],
            {"doc_id": h["doc_id"], "chunk_id": h["chunk_id"]}
        ))
    return output


In [ ]:
# Ejecución de prueba
results_weaviate = weaviate_search(query_vec, k=5)
print(f"Recuperados de Weaviate: {len(results_weaviate)} documentos.")
print(results_weaviate[0])


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [ ]:
!pip install chromadb

In [ ]:
import chromadb

# Crear cliente ligero en memoria y una colección
# Usamos get_or_create_collection para que la celda sea re-ejecutable sin
# lanzar un error si la colección ya existe de una corrida anterior.
chroma_client = chromadb.Client()
collection_chroma = chroma_client.get_or_create_collection(
    name="wikipedia_collection",
    metadata={"hnsw:space": "cosine"}
)

# Insertar elementos en Chroma por bloques debido a límites de metadatos en memoria
ids_list = [str(i) for i in chunks_df.index]
documents_list = chunks_df["text"].tolist()
metadatas_list = [{"doc_id": int(r["doc_id"]), "chunk_id": int(r["chunk_id"])} for _, r in chunks_df.iterrows()]
embeddings_list = embeddings.tolist()


In [ ]:
# Insertar en Chroma de manera limpia, por batches para evitar límites internos
batch_size = 5000
for i in range(0, len(ids_list), batch_size):
    collection_chroma.add(
        ids=ids_list[i:i+batch_size],
        embeddings=embeddings_list[i:i+batch_size],
        documents=documents_list[i:i+batch_size],
        metadatas=metadatas_list[i:i+batch_size]
    )


In [ ]:
def chroma_search(query_embedding: np.ndarray, k: int = 5):
    results = collection_chroma.query(
        query_embeddings=query_embedding.reshape(1, -1).tolist(),
        n_results=k
    )

    output = []
    # Desempaquetar la estructura interna de respuestas de Chroma
    for i in range(len(results["ids"][0])):
        output.append((
            results["ids"][0][i],
            results["distances"][0][i],
            results["documents"][0][i],
            results["metadatas"][0][i]
        ))
    return output

In [ ]:
# Una consulta con k=5
results_chroma = chroma_search(query_vec, k=5)
print("Resultado Chroma Top-1:")
print(results_chroma[0])

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [ ]:
import sqlite3
import json

# Creamos una base de datos temporal SQL que emula la consulta de vectores de pgvector
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Crear una tabla (emulando vector(D) serializando el array a texto para el cálculo)
cursor.execute('''
    CREATE TABLE documents (
        id INTEGER PRIMARY KEY,
        text TEXT,
        embedding TEXT,
        doc_id INTEGER,
        chunk_id INTEGER
    )
''')

In [ ]:
# Insertar todos los datos
insert_query = "INSERT INTO documents (id, text, embedding, doc_id, chunk_id) VALUES (?, ?, ?, ?, ?)"
data_to_sql = []
for idx, row in chunks_df.iterrows():
    data_to_sql.append((
        int(idx),
        str(row["text"]),
        json.dumps(embeddings[idx].tolist()), # Guardamos el vector
        int(row["doc_id"]),
        int(row["chunk_id"])
    ))
cursor.executemany(insert_query, data_to_sql)
conn.commit()

In [ ]:
def pgvector_search(query_embedding: np.ndarray, k: int = 5):
    q_vec_list = query_embedding.flatten()

    # Obtenemos todos los registros para realizar el ordenamiento matemático (Fórmula conceptual SQL)
    cursor.execute("SELECT id, text, embedding, doc_id, chunk_id FROM documents")
    rows = cursor.fetchall()

    scored_results = []
    for r_id, r_text, r_emb_str, r_doc_id, r_chunk_id in rows:
        db_vec = np.array(json.loads(r_emb_str))
        # Distancia coseno conceptual: 1 - (A dot B / (normA * normB))
        # Como están normalizados, se reduce a: 1 - producto_punto
        distance = 1.0 - float(np.dot(q_vec_list, db_vec))
        scored_results.append((r_id, distance, r_text, {"doc_id": r_doc_id, "chunk_id": r_chunk_id}))

    # El equivalente al ORDER BY distancia ASC LIMIT k
    scored_results.sort(key=lambda x: x[1])
    return scored_results[:k]

In [ ]:
# Ejecutar consulta SQL simulada
results_pgvector = pgvector_search(query_vec, k=5)
print(f"Top 1 obtenido vía lógica SQL (pgvector): \nDistancia: {results_pgvector[0][1]:.4f} | Texto: {results_pgvector[0][2][:120]}...")